# Day 2: Aggregation & Joins — Interactive Guide (`guideday2.ipynb`)

Welcome to **Day 2** of Week 2! Today is focused on the engine room of data transformation, business intelligence, and relational data processing: **Aggregation, Reshaping, and Merging**.

---

## Table of Contents
1. [The Split-Apply-Combine Philosophy](#1-the-split-apply-combine-philosophy)
2. [Modern Group Aggregation: Named Aggregation (`agg`)](#2-modern-group-aggregation-named-aggregation-agg)
3. [Window & Broadcast Operations: `.transform()`](#3-window--broadcast-operations-transform)
4. [Group-Level Filtering: `.filter()`](#4-group-level-filtering-filter)
5. [Reshaping Data: `pivot_table` vs `crosstab`](#5-reshaping-data-pivot_table-vs-crosstab)
6. [Relational Joins with `pd.merge()`](#6-relational-joins-with-pdmerge)
7. [Defensive Merging: The `validate` Parameter](#7-defensive-merging-the-validate-parameter)
8. [Anti-Joins & Audit Trails with `indicator=True`](#8-anti-joins--audit-trails-with-indicatortrue)
9. [Combining DataFrames: `pd.concat` & `.join()`](#9-combining-dataframes-pdconcat--join)
10. [Why Avoid `.apply()`? The Cython Fast Path & Benchmark](#10-why-avoid-apply-the-cython-fast-path)
11. [The 6 Critical Gotchas & Best Practices](#11-the-6-critical-gotchas--best-practices)
12. [20-Problem Aggregation & Joins Drill Guide](#12-20-problem-aggregation--joins-drill-guide)

## 1. The Split-Apply-Combine Philosophy

Coined by Hadley Wickham, the **Split-Apply-Combine** pattern is the core architecture behind group operations in tabular analysis:

1. **Split**: Break a `DataFrame` into smaller subsets based on the keys of one or more columns.
2. **Apply**: Execute an independent operation on each subset:
   - **Aggregation**: Reduce each group to a single summary value (e.g., `sum`, `mean`, `max`, `count`). Returns 1 row per group.
   - **Transformation**: Compute group-aware metrics broadcast back to the original group shape (e.g., group z-score, percentage of group total). Returns the exact same number of rows as the input.
   - **Filtration**: Discard entire groups based on a boolean condition evaluated on the aggregate (e.g., keep groups with at least 10 transactions).
3. **Combine**: Assemble the individual results into a single clean data structure.

```
Original DataFrame
[ Dept: A, Sales: 10 ]  ---\   Split by Dept   /--- Dept A [ 10, 30 ] -> Sum: 40  ---\ Combine
[ Dept: B, Sales: 50 ]  --->                 --->                                    ---> [ A: 40 ]
[ Dept: A, Sales: 30 ]  ---/                   \--- Dept B [ 50 ]     -> Sum: 50  ---/     [ B: 50 ]
```

In [ ]:
import pandas as pd
import numpy as np
import time

# Set display options for clean reading
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

## 2. Modern Group Aggregation: Named Aggregation (`agg`)

Prior to Pandas 0.25, aggregating multiple columns with custom target names produced messy `MultiIndex` column tuples that required awkward renaming. 

Pandas introduced **Named Aggregation** (`namedagg`), which is the modern standard for writing readable, production-grade pipelines.

### The Named Aggregation Syntax
Use keyword arguments inside `.agg()` where:
- The **keyword** is the desired new column name.
- The **value** is a tuple: `(source_column_name, aggregation_function)`.

### Built-in Fast Aggregations (String Identifiers)
Always prefer string identifiers when available (`'sum'`, `'mean'`, `'median'`, `'std'`, `'min'`, `'max'`, `'count'`, `'size'`, `'nunique'`). Pandas routes these to highly optimized C/Cython kernels.

In [ ]:
sales_df = pd.DataFrame({
    'category': ['Electronics', 'Electronics', 'Electronics', 'Apparel', 'Apparel', 'Home', 'Home'],
    'product': ['Laptop', 'Mouse', 'Monitor', 'Shirt', 'Jeans', 'Lamp', 'Chair'],
    'price': [1200.0, 35.0, 450.0, 45.0, 90.0, 35.0, 150.0],
    'quantity': [2, 10, 3, 5, 2, 4, 2],
    'rating': [4.8, 4.2, 4.7, 3.9, 4.5, 4.1, 4.6]
})
sales_df['revenue'] = sales_df['price'] * sales_df['quantity']

# Named aggregation: clean flat column names in one step!
category_summary = sales_df.groupby('category', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    avg_price=('price', 'mean'),
    max_rating=('rating', 'max'),
    product_count=('product', 'count')
)

category_summary

## 3. Window & Broadcast Operations: `.transform()`

While `.agg()` collapses $N$ rows per group into $1$ row, `.transform()` executes an operation across each group and **broadcasts the result back to the original index**.

The output of `.transform()` **always has the exact same row count and index alignment** as the input DataFrame.

### Common Use Cases:
1. **Calculating Percentage / Share of Group Total**: comparing each item to its category sum.
2. **Group Standardization (Z-Score)**: `(x - mean_group) / std_group`.
3. **Group Imputation**: filling missing values with the group median or mean.

In [ ]:
df_work = sales_df.copy()

# 1. Share of category total revenue
df_work['cat_total_revenue'] = df_work.groupby('category')['revenue'].transform('sum')
df_work['pct_of_cat_revenue'] = ((df_work['revenue'] / df_work['cat_total_revenue']) * 100).round(2)

# 2. Group Standardization (Z-Score within Category)
cat_mean = df_work.groupby('category')['price'].transform('mean')
cat_std = df_work.groupby('category')['price'].transform('std')
df_work['price_zscore_cat'] = ((df_work['price'] - cat_mean) / cat_std).round(3)

# 3. Group Imputation with Median
messy_data = pd.DataFrame({
    'dept': ['IT', 'IT', 'IT', 'HR', 'HR', 'HR'],
    'salary': [95000.0, np.nan, 105000.0, 60000.0, np.nan, 70000.0]
})
dept_median = messy_data.groupby('dept')['salary'].transform('median')
messy_data['salary_imputed'] = messy_data['salary'].fillna(dept_median)

print("--- Transform Metrics ---")
print(df_work[['category', 'product', 'price', 'pct_of_cat_revenue', 'price_zscore_cat']])
print("
--- Group Median Imputation ---")
print(messy_data)

## 4. Group-Level Filtering: `.filter()`

`.filter()` takes a function that returns a single boolean scalar for each sub-DataFrame. 
If the function evaluates to `True`, all rows belonging to that group are retained; if `False`, all rows of that group are discarded.

In [ ]:
# Keep only categories with total revenue exceeding $1,000
high_volume_cats = sales_df.groupby('category').filter(lambda g: g['revenue'].sum() > 1000)

print("Original categories:", sales_df['category'].unique())
print("Filtered categories (revenue > 1000):", high_volume_cats['category'].unique())
high_volume_cats[['category', 'product', 'revenue']]

## 5. Reshaping Data: `pivot_table` vs `crosstab`

Both tools convert long-format data into wide-format summary matrices:
- **`pivot_table`**: Best for aggregating continuous numeric values across row & column categories.
- **`crosstab`**: Best for frequency counts, contingency tables, and row/column percentages (`normalize='index'`).

In [ ]:
emp_df = pd.DataFrame({
    'dept': ['Tech', 'Tech', 'Tech', 'Sales', 'Sales', 'HR', 'HR'],
    'city': ['NY', 'SF', 'NY', 'SF', 'NY', 'NY', 'SF'],
    'salary': [120000, 140000, 115000, 80000, 85000, 70000, 75000]
})

# 1. Pivot Table with margins
pivot = emp_df.pivot_table(
    index='dept',
    columns='city',
    values='salary',
    aggfunc='mean',
    fill_value=0,
    margins=True,
    margins_name='Average_Total'
)
print("--- Pivot Table (Mean Salary) ---")
display(pivot.round(2))

# 2. Crosstab with row proportions (percentages)
ct = pd.crosstab(
    index=emp_df['dept'],
    columns=emp_df['city'],
    normalize='index'
) * 100
print("
--- Crosstab (Row Proportions %) ---")
display(ct.round(1))

## 6. Relational Joins with `pd.merge()`

`pd.merge()` provides SQL-grade relational joins:
- **Inner (`how='inner'`)**: Keys present in both tables.
- **Left (`how='left'`)**: All rows from left; fill missing right values with `NaN`.
- **Right (`how='right'`)**: All rows from right; fill missing left values with `NaN`.
- **Outer (`how='outer'`)**: Union of both tables.
- **Cross (`how='cross'`)**: Cartesian product (every left row paired with every right row).

In [ ]:
customers = pd.DataFrame({
    'cust_id': ['C01', 'C02', 'C03', 'C04'],
    'cust_name': ['Alpha Corp', 'Beta LLC', 'Gamma Inc', 'Delta Co']
})

orders = pd.DataFrame({
    'order_id': ['O101', 'O102', 'O103', 'O104'],
    'cust_id': ['C01', 'C01', 'C02', 'C99'],  # C99 has no customer entry
    'amount': [500.0, 1200.0, 350.0, 80.0]
})

# Inner Join
inner_df = pd.merge(orders, customers, on='cust_id', how='inner')
print("Inner Join (only matching IDs):")
display(inner_df)

# Left Join
left_df = pd.merge(orders, customers, on='cust_id', how='left')
print("
Left Join (preserves all orders):")
display(left_df)

## 7. Defensive Merging: The `validate` Parameter

In production pipelines, unexpected duplicates cause silent Cartesian explosion of row counts!
Always specify `validate`:
- `'1:1'`: one-to-one (keys unique on both sides).
- `'1:m'`: one-to-many (keys unique in left table).
- `'m:1'`: many-to-one (keys unique in right table).
- `'m:m'`: many-to-many (default, no check).

In [ ]:
# Defensive validation: 'many_to_one' ensures customer table has no duplicate cust_id
validated_merge = pd.merge(orders, customers, on='cust_id', how='left', validate='many_to_one')
print("Valid many_to_one merge succeeded!")

# If we mistakenly expect 'one_to_one' on orders with duplicate customer IDs:
try:
    pd.merge(orders, customers, on='cust_id', how='left', validate='one_to_one')
except pd.errors.MergeError as e:
    print("
Defensive validation caught error:", e)

## 8. Anti-Joins & Audit Trails with `indicator=True`

An **Anti-Join** identifies rows from the left table that have **no match** in the right table (e.g. customers who have never placed an order).
Pass `indicator=True` to create a `_merge` column (`'left_only'`, `'right_only'`, `'both'`).

In [ ]:
# Identify customers with ZERO orders
audit_merge = pd.merge(customers, orders, on='cust_id', how='left', indicator=True)
print("Full audit merge:")
display(audit_merge[['cust_id', 'cust_name', 'order_id', '_merge']])

# Filter for customers without orders:
zero_order_customers = audit_merge[audit_merge['_merge'] == 'left_only'][['cust_id', 'cust_name']]
print("
Anti-Join Result (Customers who never ordered):")
display(zero_order_customers)

## 9. Combining DataFrames: `pd.concat` & `.join()`

- `pd.concat([...], axis=0, ignore_index=True)`: Vertical stacking (combining time batches).
- `pd.concat([...], axis=1)`: Horizontal column-binding aligned on index.
- `df1.join(df2)`: Convenience join based on row index.

In [ ]:
jan = pd.DataFrame({'product': ['A', 'B'], 'sales': [100, 200]})
feb = pd.DataFrame({'product': ['B', 'C'], 'sales': [150, 300]})

stacked = pd.concat([jan, feb], axis=0, ignore_index=True)
print("--- Vertically Stacked (axis=0) ---")
display(stacked)

# Horizontal binding
scores = pd.DataFrame({'score': [90, 85]}, index=['E1', 'E2'])
bonuses = pd.DataFrame({'bonus': [5000, 3000]}, index=['E1', 'E2'])
horizontal = pd.concat([scores, bonuses], axis=1)
print("
--- Horizontally Bound (axis=1) ---")
display(horizontal)

## 10. Why Avoid `.apply()`? The Cython Fast Path & Benchmark

A common pitfall is using `df.groupby(...).apply(lambda ...)` for standard transformations.
Pandas built-in methods (`transform('mean')`, named aggregations) use optimized C/Cython loops, bypassing Python interpreter boxing/unboxing overhead.

In [ ]:
N = 250_000
bench_df = pd.DataFrame({
    'group': np.random.randint(1, 150, size=N),
    'val': np.random.randn(N)
})

# 1. Slow .apply with lambda
t0 = time.perf_counter()
res_apply = bench_df.groupby('group')['val'].apply(lambda x: x - x.mean()).values
t_apply = time.perf_counter() - t0

# 2. Fast vectorized .transform
t0 = time.perf_counter()
res_transform = (bench_df['val'] - bench_df.groupby('group')['val'].transform('mean')).values
t_transform = time.perf_counter() - t0

print(f"Benchmark on {N:,} rows:")
print(f"  .apply(lambda ...) time:   {t_apply:.4f} s")
print(f"  .transform('mean') time:  {t_transform:.4f} s")
print(f"  -> Speedup:                {t_apply / t_transform:.1f}x faster!")

## 11. The 6 Critical Gotchas & Best Practices

1. **`dropna=True` in `groupby` drops missing keys**: Always specify `dropna=False` if missing keys are meaningful.
2. **Duplicate keys cause Cartesian explosion in merges**: Enforce `validate='m:1'` or `'1:m'` to catch unexpected duplicates.
3. **MultiIndex columns after `.agg()`**: Prefer named aggregation (`agg(new_col=('col', 'func'))`) to keep column headers flat.
4. **Index alignment in `.transform()`**: Ensure the index is unique when broadcasting series back to the original table.
5. **Categorical dtypes in crosstabs**: Specify `observed=True` if you do not want unobserved categories shown with 0 counts.
6. **In-place column collision in merge**: Check `suffixes=('_left', '_right')` to avoid confusing `_x` and `_y` headers.

## 12. 20-Problem Aggregation & Joins Drill Guide

Now that you've mastered the concepts:
1. Open [`drill_blank_practice.py`](file:///c:/Users/ASUS/Desktop/ds-internship-2026/W2/d2/drill_blank_practice.py) and solve each drill from scratch.
2. Run test checks via terminal:
   ```bash
   python drill_blank_practice.py
   ```
3. Check the complete verified solutions in [`drill_20_problems.py`](file:///c:/Users/ASUS/Desktop/ds-internship-2026/W2/d2/drill_20_problems.py).